# What drives the price of a car?

![](images/kurt.jpeg)

**OVERVIEW**

In this application, you will explore a dataset from Kaggle. The original dataset contained information on 3 million used cars. The provided dataset contains information on 426K cars to ensure speed of processing.  Your goal is to understand what factors make a car more or less expensive.  As a result of your analysis, you should provide clear recommendations to your client -- a used car dealership -- as to what consumers value in a used car.

### CRISP-DM Framework

<center>
    <img src = images/crisp.png width = 50%/>
</center>


To frame the task, throughout our practical applications, we will refer back to a standard process in industry for data projects called CRISP-DM.  This process provides a framework for working through a data problem.  Your first step in this application will be to read through a brief overview of CRISP-DM [here](https://mo-pcco.s3.us-east-1.amazonaws.com/BH-PCMLAI/module_11/readings_starter.zip).  After reading the overview, answer the questions below.

### Business Understanding

From a business perspective, we are tasked with identifying key drivers for used car prices.  In the CRISP-DM overview, we are asked to convert this business framing to a data problem definition.  Using a few sentences, reframe the task as a data task with the appropriate technical vocabulary. 

The objective is to model and predict used-car prices, a continuous target variable, by leveraging multiple explanatory features. The aim is to conduct exploratory data analysis (EDA) and feature assessment to determine which independent variables exhibit the strongest statistical association with price. This process may involve techniques like correlation analysis, feature engineering, and various regression approaches (e.g., linear or regularized regression) to enhance model accuracy and interpretability.

### Data Understanding

After considering the business understanding, we want to get familiar with our data.  Write down some steps that you would take to get to know the dataset and identify any quality issues within.  Take time to get to know the dataset and explore what information it contains and how this could be used to inform your business understanding.

To assess the dataset and spot potential quality issues, I would follow a structured process inspired by the CRISP-DM framework.

First, I would load the data and review its structure—looking at rows, columns, data types, and key features such as mileage, year, and manufacturer. Next, I would check summary statistics for numeric columns to catch outliers or unrealistic values, like negative mileage. I would also identify missing values and consider whether to clean or impute them. Checking for duplicate records ensures the analysis reflects real market conditions. Analyzing distributions of variables like price and mileage helps reveal outliers, trends, and how features affect pricing. I would also review categorical variables for inconsistencies and to understand which brands or types are most common. Detecting outliers and invalid values, like impossible years or zero prices, improves reliability. Exploring relationships between features and price helps uncover what drives used-car prices. Finally, I would focus on features most relevant to predicting price, ensuring the analysis targets what consumers value.

By following these steps, I can quickly assess data quality, spot key issues, and prepare the dataset for meaningful analysis.



In [1]:
import pandas as pd
vehicles = pd.read_csv("data/vehicles.csv")

In [2]:
vehicles.head()

,id,region,price,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,VIN,drive,size,type,paint_color,state
0,7222695916,prescott,6000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,az
1,7218891961,fayetteville,11900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ar
2,7221797935,florida keys,21000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fl
3,7222270760,worcester / central MA,1500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ma
4,7210384030,greensboro,4900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nc


In [3]:
vehicles.tail()

,id,region,price,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,VIN,drive,size,type,paint_color,state
426875,7301591192,wyoming,23590,2019.0,nissan,maxima s sedan 4d,good,6 cylinders,gas,32226.0,clean,other,1N4AA6AV6KC367801,fwd,NaN,sedan,NaN,wy
426876,7301591187,wyoming,30590,2020.0,volvo,s60 t5 momentum sedan 4d,good,NaN,gas,12029.0,clean,other,7JR102FKXLG042696,fwd,NaN,sedan,red,wy
426877,7301591147,wyoming,34990,2020.0,cadillac,xt4 sport suv 4d,good,NaN,diesel,4174.0,clean,other,1GYFZFR46LF088296,NaN,NaN,hatchback,white,wy
426878,7301591140,wyoming,28990,2018.0,lexus,es 350 sedan 4d,good,6 cylinders,gas,30112.0,clean,other,58ABK1GG4JU103853,fwd,NaN,sedan,silver,wy
426879,7301591129,wyoming,30590,2019.0,bmw,4 series 430i gran coupe,good,NaN,gas,22716.0,clean,other,WBA4J1C58KBM14708,rwd,NaN,coupe,NaN,wy


In [4]:
vehicles.describe()

,id,price,year,odometer
count,4.268800e+05,4.268800e+05,425675.000000,4.224800e+05
mean,7.311487e+09,7.519903e+04,2011.235191,9.804333e+04
std,4.473170e+06,1.218228e+07,9.452120,2.138815e+05
min,7.207408e+09,0.000000e+00,1900.000000,0.000000e+00
25%,7.308143e+09,5.900000e+03,2008.000000,3.770400e+04
50%,7.312621e+09,1.395000e+04,2013.000000,8.554800e+04
75%,7.315254e+09,2.648575e+04,2017.000000,1.335425e+05
max,7.317101e+09,3.736929e+09,2022.000000,1.000000e+07


In [5]:
vehicles.isnull().sum()

id                   0
region               0
price                0
year              1205
manufacturer     17646
model             5277
condition       174104
cylinders       177678
fuel              3013
odometer          4400
title_status      8242
transmission      2556
VIN             161042
drive           130567
size            306361
type             92858
paint_color     130203
state                0
dtype: int64

### Data Preparation

After our initial exploration and fine-tuning of the business understanding, it is time to construct our final dataset prior to modeling.  Here, we want to make sure to handle any integrity issues and cleaning, the engineering of new features, any transformations that we believe should happen (scaling, logarithms, normalization, etc.), and general preparation for modeling with `sklearn`. 

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error, r2_score

vehicles = pd.read_csv("data/vehicles.csv")

In [7]:
# -------------------------
# Cleaning the data
# -------------------------

# Remove duplicates
vehicles = vehicles.drop_duplicates()

# Remove invalid prices
vehicles = vehicles[(vehicles["price"] > 100) & (vehicles["price"] < 250000)]

# Remove unrealistic years
vehicles = vehicles[(vehicles["year"] >= 1980) & (vehicles["year"] <= 2026)]

# Remove unrealistic odometer values
vehicles = vehicles[(vehicles["odometer"] >= 0) & (vehicles["odometer"] <= 500000)]

# Feature engineering: car age
vehicles["car_age"] = 2026 - vehicles["year"]

# Drop useless columns
vehicles = vehicles.drop(columns=["id", "VIN"])

In [ ]:
# -------------------------
# Defining features
# -------------------------

# Target variable for prediction
target = "price"

# List of numerical features that will be used in the model
numeric_features = [
    "odometer",     # Vehicle mileage
    "car_age"       # Age of the vehicle in years
]

# List of categorical features that will be encoded for the model
categorical_features = [
    "region",       # Geographic region where the car is being sold
    "manufacturer", # Car manufacturer (brand)
    "model",        # Specific model of the car
    "condition",    # Condition of the vehicle (e.g., excellent, good, fair)
    "cylinders",    # Number of cylinders in the engine
    "fuel",         # Fuel type (e.g., gas, diesel, hybrid)
    "title_status", # Status of the car title (e.g., clean, salvage)
    "transmission", # Transmission type (e.g., automatic, manual)
    "drive",        # Drive type (e.g., fwd, rwd, 4wd)
    "size",         # Size of the vehicle (e.g., compact, mid-size, full-size)
    "type",         # Body type (e.g., sedan, SUV, truck)
    "paint_color",  # Exterior color of the vehicle
    "state"         # State where the car is being sold
]

In [ ]:
# -------------------------
# Preprocessing pipelines
# -------------------------

# Pipeline for numerical features:
# 1. Replace missing values with median (robust to outliers)
# 2. Standardize features to zero mean and unit variance
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Pipeline for categorical features:
# 1. Replace missing values with most frequent category
# 2. Convert categories to one-hot encoded vectors
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))  # 'ignore' handles categories not seen during fit
])

# Combine both pipelines into a single transformer
# Each pipeline will be applied to its respective column types
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),     # Apply numeric pipeline to numeric columns
    ("cat", categorical_pipeline, categorical_features)  # Apply categorical pipeline to categorical columns
])

In [ ]:
# -------------------------
# Full pipeline with model
# -------------------------

# Initialize Ridge regression model with regularization strength alpha=10
# Ridge regression helps prevent overfitting by penalizing large coefficients
model = Ridge(alpha=10)

# Create a complete pipeline that combines preprocessing and model training
# This ensures consistent application of preprocessing steps to both training and test data
pipeline = Pipeline([
    ("preprocess", preprocessor),  # Apply the preprocessing steps defined earlier
    ("model", model)               # Apply the Ridge regression model
])


# Log transform price for better regression performance
# This helps normalize skewed price distributions and stabilize variance
final_model = TransformedTargetRegressor(
    regressor=pipeline,            # Use our preprocessing+model pipeline
    func=np.log1p,                 # Transform target using log(1+x) before training
    inverse_func=np.expm1          # Transform predictions back using exp(x)-1
)

In [ ]:
# -------------------------
# Train and test split
# -------------------------

# Separate features (X) and target variable (y)
X = vehicles.drop(columns=[target])  # Remove target column from features
y = vehicles[target]                 # Extract target variable

# Split data into training (80%) and testing (20%) sets
# random_state ensures reproducibility of the split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,    # 20% of data used for testing
    random_state=42   # Set seed for reproducible results
)


# -------------------------
# Train the model
# -------------------------

# Fit the final model to the training data
# This trains the model by finding patterns between X_train features and y_train target values
final_model.fit(X_train, y_train)

In [12]:
# -------------------------
# Make predictions
# -------------------------

predictions = final_model.predict(X_test)


# -------------------------
# 9. Evaluate Model
# -------------------------

print("Mean Absolute Error:", mean_absolute_error(y_test, predictions))
print("R² Score:", r2_score(y_test, predictions))


Mean Absolute Error: 4721.359821562249
R² Score: 0.6993770024370547


### Modeling

With your (almost?) final dataset in hand, it is now time to build some models.  Here, you should build a number of different regression models with the price as the target.  In building your models, you should explore different parameters and be sure to cross-validate your findings.

In [13]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, GridSearchCV, KFold

In [ ]:
# Create a KFold cross-validator with 5 splits (folds)
# shuffle=True randomizes the data before splitting
# random_state=42 ensures reproducible results across runs
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
#-----------------------
# Creating plot to use
#-----------------------
def plot_regression_results(model, X_test, y_test, model_name="Regression Model"):
    """
    Creates a scatter plot comparing actual vs predicted values for regression models.
    
    Parameters:
    -----------
    model : estimator object
        The trained regression model with a predict method
    X_test : array-like
        Test features to generate predictions
    y_test : array-like
        True target values for comparison
    model_name : str, default="Regression Model"
        Name of the model to display in the plot title
    """
    
    # Generate predictions from the model using test features
    y_pred = model.predict(X_test)
    
    # Create a new figure for the plot
    plt.figure()
    
    # Create scatter plot of actual vs predicted values with transparency
    plt.scatter(y_test, y_pred, alpha=0.3)
    
    # Add a diagonal line representing perfect predictions (y=x)
    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    plt.plot([min_val, max_val], [min_val, max_val])
    
    # Add descriptive labels and title to the plot
    plt.xlabel("Actual Price")
    plt.ylabel("Predicted Price")
    plt.title(f"{model_name}: Predicted vs Actual Prices")
    
    # Display the plot
    plt.show()

In [ ]:
# -------------------------
# Linear Regression
# -------------------------
# Create a linear regression model with log transformation of the target variable
# TransformedTargetRegressor applies transformation to the target variable before fitting
# and reverses the transformation for predictions
linear_model = TransformedTargetRegressor(
    regressor=Pipeline([
        ("preprocess", preprocessor),  # Apply the preprocessing steps defined earlier
        ("model", LinearRegression())  # Use standard linear regression algorithm
    ]),
    func=np.log1p,      # Transform target using log(1+x) to handle skewed data
    inverse_func=np.expm1  # Inverse transform using exp(x)-1 when predicting
)

# Evaluate model performance using cross-validation
# neg_mean_absolute_error is used because scikit-learn converts all metrics to higher_is_better
linear_scores = cross_val_score(
    linear_model,
    X_train,
    y_train,
    cv=kf,  # Use the predefined KFold cross-validation strategy
    scoring="neg_mean_absolute_error"  # Evaluate using Mean Absolute Error
)

# Train the final model on the entire training dataset
linear_model.fit(X_train, y_train)

# Visualize the model's predictions against actual values
plot_regression_results(
    linear_model,
    X_test,
    y_test,
    "Linear Regression"
)

# Display the average MAE from cross-validation
# Negate the score since cross_val_score returns negative MAE
print("Linear Regression MAE:", -linear_scores.mean())

In [ ]:
# -------------------------
# Ridge Regression
# -------------------------
# Create a pipeline that first preprocesses the data and then applies Ridge Regression
ridge_pipeline = Pipeline([
    ("preprocess", preprocessor),  # Apply the preprocessing steps defined earlier
    ("model", Ridge())             # Apply Ridge Regression algorithm
])

# Wrap the pipeline in a TransformedTargetRegressor to apply log transformation to the target variable
# This helps handle skewed target distributions and often improves model performance
ridge_model = TransformedTargetRegressor(
    regressor=ridge_pipeline,      # The regression pipeline to use
    func=np.log1p,                 # Transform target using log(1+x) before fitting
    inverse_func=np.expm1          # Transform predictions back using exp(x)-1
)

# Define hyperparameters to tune
# Alpha is the regularization strength - higher values increase regularization
ridge_params = {
    "regressor__model__alpha": [0.1, 1, 10, 50, 100]
}

# Set up grid search with cross-validation to find the best alpha value
ridge_grid = GridSearchCV(
    ridge_model,                   # Model to tune
    ridge_params,                  # Parameter grid to search
    cv=kf,                         # Cross-validation strategy defined earlier
    scoring="neg_mean_absolute_error",  # Evaluation metric (negative because GridSearchCV maximizes)
    n_jobs=-1                      # Use all available CPU cores
)

# Train the model on the training data
ridge_grid.fit(X_train, y_train)

# Visualize the model's performance on test data
plot_regression_results(
    ridge_grid.best_estimator_,    # Use the best model found by grid search
    X_test,
    y_test,
    "Ridge Regression"             # Title for the plot
)

# Print the best hyperparameters and corresponding performance
print("Best Ridge alpha:", ridge_grid.best_params_)
print("Best Ridge MAE:", -ridge_grid.best_score_)  # Negate to get positive MAE

In [ ]:
# -------------------------
# Lasso Regression
# -------------------------
# Create a pipeline that first preprocesses the data and then applies Lasso regression
# max_iter is set high to ensure convergence
lasso_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", Lasso(max_iter=10000))
])

# Wrap the pipeline in a TransformedTargetRegressor to apply log transformation to the target variable
# This helps handle skewed target distributions and stabilize variance
lasso_model = TransformedTargetRegressor(
    regressor=lasso_pipeline,
    func=np.log1p,      # Apply log(1+x) transformation to target before fitting
    inverse_func=np.expm1  # Apply exp(x)-1 to predictions to get back original scale
)

# Define hyperparameter grid for alpha (regularization strength)
# Higher alpha values increase regularization and sparsity
lasso_params = {
    "regressor__model__alpha": [0.001, 0.01, 0.1, 1, 10]
}

# Set up grid search with cross-validation to find optimal alpha value
# Using negative MAE as scoring metric (GridSearchCV maximizes score)
lasso_grid = GridSearchCV(
    lasso_model,
    lasso_params,
    cv=kf,  # Use predefined K-fold cross-validation
    scoring="neg_mean_absolute_error",
    n_jobs=-1  # Use all available CPU cores
)

# Train the model with grid search
lasso_grid.fit(X_train, y_train)

# Visualize the model's performance on test data
plot_regression_results(
    lasso_grid.best_estimator_,
    X_test,
    y_test,
    "Lasso Regression"
)

# Print the best hyperparameter and corresponding performance metric
print("Best Lasso alpha:", lasso_grid.best_params_)
print("Best Lasso MAE:", -lasso_grid.best_score_)

In [ ]:
# -------------------------
# Random Forest Regression
# -------------------------
# Create a pipeline that first preprocesses the data and then applies Random Forest Regression
rf_pipeline = Pipeline([
    ("preprocess", preprocessor),  # Apply the preprocessing steps defined earlier
    ("model", RandomForestRegressor(random_state=42))  # Initialize Random Forest model with fixed random state for reproducibility
])

# Wrap the pipeline in a TransformedTargetRegressor to apply log transformation to the target variable
# This helps handle skewed target distributions and often improves model performance
rf_model = TransformedTargetRegressor(
    regressor=rf_pipeline,  # Use the pipeline defined above
    func=np.log1p,  # Apply log(1+x) transformation to target before training
    inverse_func=np.expm1  # Apply exp(x)-1 to predictions to get back original scale
)

# Define hyperparameter grid for Random Forest tuning
rf_params = {
    "regressor__model__n_estimators": [50, 100],  # Test different numbers of trees
    "regressor__model__max_depth": [10, 20, None],  # Test different tree depths (None = unlimited)
    "regressor__model__min_samples_split": [2, 5]  # Minimum samples required to split a node
}

# Set up grid search with cross-validation to find optimal hyperparameters
rf_grid = GridSearchCV(
    rf_model,  # Model to tune
    rf_params,  # Parameter grid
    cv=kf,  # Cross-validation strategy defined earlier
    scoring="neg_mean_absolute_error",  # Optimize for MAE (negative because GridSearchCV maximizes)
    n_jobs=-1  # Use all available CPU cores
)

# Train the model with grid search
rf_grid.fit(X_train, y_train)

# Visualize model performance on test data
plot_regression_results(
    rf_grid.best_estimator_,  # Use the best model found by grid search
    X_test,
    y_test,
    "Random Forest Regression"
)

# Print the best hyperparameters and corresponding MAE score
print("Best Random Forest params:", rf_grid.best_params_)
print("Best Random Forest MAE:", -rf_grid.best_score_)  # Negate to convert back to positive MAE

In [ ]:
# --------------------------
# Compare Model Performance
# --------------------------
# Create a DataFrame to compare the performance of different regression models
# The negative MAE scores are converted back to positive values for easier interpretation
results = pd.DataFrame({
    "Model": ["Linear", "Ridge", "Lasso", "Random Forest"],  # List of models being compared
    "MAE": [
        -linear_scores.mean(),        # Mean MAE from cross-validation for Linear Regression
        -ridge_grid.best_score_,      # Best MAE score from Ridge Regression grid search
        -lasso_grid.best_score_,      # Best MAE score from Lasso Regression grid search
        -rf_grid.best_score_          # Best MAE score from Random Forest grid search
    ]
})

# Display the models sorted by MAE (lower is better)
print(results.sort_values("MAE"))

In [ ]:
#----------------------------------
# Evaluate Best Model on Test Data
# ---------------------------------
best_model = rf_grid.best_estimator_

predictions = best_model.predict(X_test)

print("Test MAE:", mean_absolute_error(y_test, predictions))
print("Test R2:", r2_score(y_test, predictions))


### Evaluation

With some modeling accomplished, we aim to reflect on what we identify as a high-quality model and what we are able to learn from this.  We should review our business objective and explore how well we can provide meaningful insight into drivers of used car prices.  Your goal now is to distill your findings and determine whether the earlier phases need revisitation and adjustment or if you have information of value to bring back to your client.

#### Review of Business Objective

The original business objective was to identify the key factors influencing used-car prices, enabling the dealership to make better decisions on vehicle acquisition, pricing, and inventory management. To address this goal from a data science perspective, the problem was framed as a supervised regression task. The aim was to predict a vehicle's price based on features such as mileage, age, manufacturer, condition, fuel type, and other relevant vehicle characteristics.

To solve the problem, multiple regression models were developed and rigorously evaluated using cross-validation techniques. This approach ensured that the results were both reliable and generalizable to unseen data. The models tested included Linear Regression, Ridge Regression, Lasso Regression, and Random Forest Regression, providing a diverse set of methodologies for comparison.

#### Identification of a High-Quality Model

A high-quality model was defined based on the following criteria:

* Low Mean Absolute Error (MAE), meaning predictions are close to actual vehicle prices
* High R² score, indicating strong explanatory power
* Consistent performance across cross-validation folds
* Ability to capture nonlinear relationships between vehicle attributes and price
* Interpretability and usefulness for business decision-making

Among the models tested, the Random Forest Regression model demonstrated the best overall performance. Unlike linear models, Random Forests were able to capture complex, nonlinear relationships between features and vehicle price that could not be fully represented otherwise. Results from cross-validation revealed that Random Forest produced the lowest prediction error and delivered the most consistent and reliable performance across different data subsets.

These findings suggest that used-car prices are influenced by nonlinear combinations of factors rather than simple linear relationships. As a result, models capable of capturing this complexity, such as Random Forests, are better suited to accurately predict vehicle prices.

### Deployment

Now that we've settled on our models and findings, it is time to deliver the information to the client.  You should organize your work as a basic report that details your primary findings.  Keep in mind that your audience is a group of used car dealers interested in fine-tuning their inventory.

### Executive Summary

This analysis examined over 426,000 used-vehicle listings to identify the primary factors that influence used-car prices. Multiple predictive models were developed and evaluated to understand how vehicle characteristics affect resale value. The analysis found that the most important factors influencing vehicle price are vehicle age, mileage, manufacturer and model, vehicle condition, and vehicle type (such as SUV, truck, or sedan). Using these findings, dealerships can optimize inventory acquisition, improve pricing strategies, and maximize profitability. The Random Forest regression model provided the most accurate predictions, confirming that used car prices are influenced by a combination of vehicle attributes rather than a single factor.

### Business Objective

The goal of this project was to determine which vehicle characteristics most strongly influence used car prices. This information can help dealerships purchase vehicles with higher resale value, avoid low-profit inventory, price vehicles competitively, and improve overall profitability.

### Data Overview

The dataset included approximately 426,000 used vehicle listings and contained the following key information: price, manufacturer and model, year, mileage (odometer reading), vehicle condition, fuel type, transmission, vehicle type, and location. This large dataset provides a reliable view of trends in the used-vehicle market.

### Modeling Approach

Several regression models were developed and tested, including Linear Regression, Ridge Regression, Lasso Regression, and Random Forest Regression. Each model was evaluated using cross-validation to ensure reliable performance. The Random Forest model performed best because it can capture complex relationships between vehicle characteristics and price.

### Key Findings: What Drives Used Car Prices

Vehicle age is the most important factor. Newer vehicles consistently sell for significantly higher prices, while older vehicles depreciate rapidly, especially after 10 years. Dealerships should prioritize purchasing vehicles that are 8–10 years old or less.

Mileage strongly affects value. Vehicles with lower mileage command much higher prices, while high-mileage vehicles lose value significantly. Dealerships should focus on acquiring vehicles with mileage below 100,000 miles whenever possible.

Manufacturer and brand reputation matter. Certain manufacturers, such as Toyota, Honda, Ford (especially trucks), Chevrolet, and Subaru, consistently maintain higher resale values. Luxury brands may also have high value but can carry a higher risk. Dealerships should prioritize reliable, high-demand brands with strong resale value.

Vehicle condition significantly impacts price. Vehicles in excellent or good condition sell for much higher prices, while those in poor condition sell at significant discounts. Investing in minor repairs and reconditioning before resale is recommended.

Vehicle type influences demand and price. SUVs and trucks generally sell for higher prices than sedans and retain value better due to higher demand. Increasing the inventory of SUVs and trucks can improve profitability.

### Model Performance Summary

The Random Forest regression model provided the best prediction accuracy. This confirms that used-car pricing depends on a combination of factors, including age, mileage, brand, condition, and vehicle type. The model can be used to estimate fair vehicle prices and identify high-value inventory opportunities.

### Business Recommendations

Based on the analysis, the dealership should prioritize acquiring vehicles with the following characteristics: low mileage (under 100,000 miles), newer vehicles (under 10 years old), reliable manufacturers (Toyota, Honda, Ford, Subaru), excellent or good condition, and SUVs or trucks. These vehicles offer the best balance of resale value and profitability. Dealerships should approach high-mileage vehicles (over 150,000 miles), older vehicles (over 15 years old), poor-condition vehicles, and low-demand brands with caution, as these carry higher risk and lower profit margins.

### Business Impact

By applying these findings, the dealership can improve inventory quality, increase average profit per vehicle, reduce the risk of overpaying for inventory, price vehicles more accurately, and increase sales efficiency. This data-driven approach improves decision-making and provides a competitive advantage.

### Opportunities for Future Improvement

Future analysis could include additional data such as accident history, maintenance records, trim level, and market demand trends. Incorporating these factors could further improve pricing accuracy and dealership performance.

### Conclusion

This analysis successfully identified the key factors that influence used car prices. Vehicle age, mileage, manufacturer, condition, and vehicle type were the most important predictors of value. By prioritizing high-value vehicle characteristics, the dealership can optimize inventory selection, improve pricing accuracy, and maximize profitability. The Random Forest regression model provides a reliable tool for predicting vehicle prices and supporting data-driven business decisions.

By prioritizing high-value vehicle characteristics, the dealership can optimize inventory selection, improve pricing accuracy, and maximize profitability.

The Random Forest regression model provides a reliable tool for predicting vehicle prices and supporting data-driven business decisions.